# Display Nonlinearity — How does the monitor interpret the values we send?

In this notebook we explore how a monitor's nonlinear response (gamma) affects image brightness,
and how the sRGB standard describes this nonlinearity mathematically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import HdM as HdM # a useful function I prepared for you to display images pixel per pixel on your monitor

## Part 1 — Match a solid grey to a 50% checkerboard

The image below shows **50% checkerboard** on top.

Try to adjust the gray on the bottom via the +/- buttons to match the checkerboard as precise as possible. Lean back or take off your glasses for the checkerbaord to mix as good as possible to gray.

In [ ]:
w = 512 # Width of the checkerboard patch. Adjust to smaller values on lower resolution monitors
h = 256 # Height of the checkerboard patch. Adjust to smaller values on lower resolution monitors

checker = np.tile([[0, 1], [1, 0]], (h//2, w//2)).astype(float)

result = HdM.show2(checker, start_color=0.5)


In [ ]:
print(f"Your match: {result[0]:.3f}")
print('\nWhy is the matching value not 0.5, even if 50% of the pixels are turned on?')
print('\nCan you explain?')

## Part 2 — Measure your full monitor curve

We create dithered patches for a range of physical grey levels 1, 3/4, 1/2, 6/16, 1/4, 3/16, 2/16, 1/16, 1/32, 1/64 and match each one with a solid grey value.
The resulting curve reveals the monitor's nonlinearity.

In [ ]:
# These are the luminance realtive to the peak white of the monitors we will create dither pattern for
reference_colors = np.array([1, 3/4, 1/2, 6/16, 1/4, 3/16, 2/16, 1/16, 1/32, 1/64, 0])

# We will try to find out which value needs to be sent to the monitor to match the reference luminances
matched_colors = np.array([1, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0])

# Now it's your turn: 
myDitheredPatch = np.tile( ___ ).astype(float)  # Hinweis: How can you dither 3/4 or 75% peak luminace? Hint: Look at the way I created the 'checker' variable in the cell above.

# Display the 3/4 Patch and match:
result = HdM.show2(myDitheredPatch, start_color=matched_colors[1])

In [ ]:
print(f"Your match of 3/4 was: {result[0]:.3f}")
matched_colors[1] = result[0]

# The next luminance to match is 1/2
myDitheredPatch = np.tile( ___ ).astype(float)  # Hinweis: How can you dither 1/2 or 50% peak luminace?

# Display the 1/2 Patch and match:
result = HdM.show2(myDitheredPatch, start_color=matched_colors[2])

## Part 4 — Plot measured curve vs. sRGB standard

The sRGB nonlinearity is defined as:

$$L_{\text{linear}} = \begin{cases}
\left(\frac{V + 0.055}{1.055}\right)^{2.4} & \text{if } V > 0.04045 \\
\frac{V}{12.92} & \text{otherwise}
\end{cases}$$

where $V$ is the encoded signal value (0…1) and $L$ is the linear light output.

In [ ]:
# sRGB transfer functions
sRGB2linear = lambda x: np.where(x > 0.04045, ((x + 0.055) / 1.055) ** 2.4, x / 12.92)
linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * x ** (1/2.4) - 0.055, 12.92 * x)

ref_colors = np.array([0, 1/64, 2/64, 1/16, 2/16, 3/16, 1/4, 6/16, 2/4, 3/4, 1.0])

x = np.linspace(0, 1, 500)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(test_colors, ref_colors, 'o-', lw=2, label='Measured (color matching)')
ax.plot(x, sRGB2linear(x), ':', lw=2, label='sRGB standard')
ax.set_xlabel('Monitor input value (encoded)')
ax.set_ylabel('Linear light output')
ax.set_title('Monitor nonlinearity')
ax.legend()
ax.grid(True)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Part 5 — Effect of monitor nonlinearity on images

We download a test image encoded in sRGB and show it:
1. As-is (correct for a standard sRGB monitor)
2. Linearized and then re-encoded for the measured monitor curve

Toggle between the two to see the difference.

In [ ]:
import urllib.request
from PIL import Image
import io

url = 'https://cdn.shopify.com/s/files/1/2691/1130/files/ARRIColorTool_grande.jpg?v=1524533887'
with urllib.request.urlopen(url) as r:
    img_sRGB = np.array(Image.open(io.BytesIO(r.read())).convert('RGB')) / 255.0

img_sRGB = img_sRGB[60:240, 360:, :]

# Linearize, then re-encode with the measured curve
img_lin = sRGB2linear(img_sRGB)
img_monitor = np.clip(
    np.interp(img_lin.ravel(), ref_colors, test_colors).reshape(img_lin.shape), 0, 1
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(img_sRGB)
axes[0].set_title('Standard sRGB display')
axes[0].axis('off')
axes[1].imshow(img_monitor)
axes[1].set_title('Corrected for measured monitor curve')
axes[1].axis('off')
plt.tight_layout()
plt.show()